# Improving the Model with Data Augmentation

Our baseline model showed signs of overfitting. To address this, we'll use data augmentation to generate more training data and improve the model's ability to generalize.

### 1. Import Libraries

In [4]:
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

## 2. Load Data

In [ ]:
X_train, y_train = np.load('../results/X_train.npy'), np.load('../results/y_train.npy')
X_test, y_test = np.load('../results/X_test.npy'), np.load('../results/y_test.npy')
history_data = np.load('../results/training_history.npy', allow_pickle=True).item()

### 3. Data Augmentation

To combat overfitting, we'll use data augmentation to artificially expand our training dataset. An `ImageDataGenerator` will be configured to apply random transformations like rotations, shifts, zooms, and flips to the training images. This process helps the model learn from a more diverse set of examples, making it more robust and less likely to memorize the training data. By exposing the model to these variations, we encourage it to learn the underlying patterns of brain tumors rather than superficial features of specific images. This should improve its ability to generalize to new, unseen data.

In [9]:
X_train = np.repeat(X_train, 3, axis=-1)
X_val = np.repeat(X_test, 3, axis=-1)
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

### 4. Load and Re-train the Model

Now, we'll load the baseline model and re-train it using the augmented data. We'll also use early stopping to prevent overfitting and save the best model.

In [ ]:
model = load_model('../results/baseline_model.keras')
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('../results/best_model.keras', save_best_only=True, monitor='val_loss')
model.fit(datagen.flow(X_train, y_train, batch_size=32), epochs=50, validation_data=(X_val, y_test), callbacks=[early_stopping, model_checkpoint])